<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<span style="display:inline-block;margin-bottom:10px;padding:2px 12px;border-radius:999px;background:#6366f1;color:#fff;font:600 11px/20px system-ui,sans-serif;letter-spacing:.05em;text-transform:uppercase">Step 3</span>
<h1 style="font-size:1.35em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">Step 3: Storage and business rules</h1>
<p style="margin:.55em 0">Rules are plain functions over a database path, free of model and transport concerns, so they are easy to test and
to port. These follow DocAssist's registration and appointment code.</p>

</div>

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<h2 style="font-size:1.2em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">Storage</h2>
<p style="margin:.55em 0">SQLite with production habits: one connection per operation (skills run on worker threads), WAL mode so readers and a
writer don't block each other, a busy timeout, foreign keys on, a schema version, and demo data only when config asks.</p>

</div>

In [1]:
%%writefile build/app/store.py
"""Storage: SQLite, one connection per operation (skills run on worker threads; connections don't cross threads).

Production notes: WAL mode lets readers and one writer work at once; busy_timeout makes a second writer wait
instead of failing. Writes that check-then-insert use BEGIN IMMEDIATE so two callers can't take one slot.
Swapping in Postgres means re-implementing db() and the SQL dialect bits; the rules never open connections
themselves. Table shapes follow DocAssist (pyDocAssist table/Patient.py, appointment_api.py).
"""
import contextlib
import sqlite3
from datetime import date, timedelta
from pathlib import Path

SCHEMA_VERSION = 1
SCHEMA = """
CREATE TABLE IF NOT EXISTS Meta (key TEXT PRIMARY KEY, value TEXT NOT NULL);
CREATE TABLE IF NOT EXISTS Patient (
    Patient_ID INTEGER PRIMARY KEY AUTOINCREMENT,
    First_Name TEXT NOT NULL, Last_Name TEXT NOT NULL,
    DOB TEXT NOT NULL, age INTEGER, sex TEXT,
    Phone_Number TEXT NOT NULL, Email TEXT,
    is_active INTEGER NOT NULL DEFAULT 1,
    created_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP);
CREATE INDEX IF NOT EXISTS patient_phone ON Patient(Phone_Number);
CREATE TABLE IF NOT EXISTS Doctor (
    Doctor_ID INTEGER PRIMARY KEY AUTOINCREMENT,
    First_Name TEXT NOT NULL, Last_Name TEXT NOT NULL, Specialty TEXT NOT NULL);
CREATE TABLE IF NOT EXISTS Availability (
    doctor_id INTEGER NOT NULL REFERENCES Doctor(Doctor_ID),
    day TEXT NOT NULL, start_time TEXT NOT NULL, end_time TEXT NOT NULL,
    duration_minutes INTEGER NOT NULL DEFAULT 15);
CREATE TABLE IF NOT EXISTS Appointment (
    Appointment_ID INTEGER PRIMARY KEY AUTOINCREMENT,
    patient_id INTEGER REFERENCES Patient(Patient_ID),      -- NULL for a doctor's leave block
    doctor_id INTEGER NOT NULL REFERENCES Doctor(Doctor_ID),
    appointment_date TEXT NOT NULL,                           -- YYYY-MM-DD HH:MM:00
    status TEXT NOT NULL DEFAULT 'scheduled',                 -- scheduled | cancelled | completed | unavailable
    reason TEXT, notes TEXT,
    created_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP);
CREATE INDEX IF NOT EXISTS appt_doctor_day ON Appointment(doctor_id, appointment_date);
CREATE INDEX IF NOT EXISTS appt_patient ON Appointment(patient_id, status);
"""

DEMO_PATIENTS = [("asha", "sharma", "1985-03-02", 41, "Female", "919812345678"),
                 ("rohan", "sharma", "2015-07-19", 11, "Male", "919812345678"),  # a child shares his mother's phone
                 ("vikram", "nair", "1979-11-30", 46, "Male", "919845012367")]
DEMO_DOCTORS = [("anil", "kapoor", "General Medicine",
                 [(d, "09:00", "13:00", 15) for d in ("Monday", "Tuesday", "Wednesday", "Thursday", "Friday")]),
                ("sunita", "rao", "Pediatrics", [(d, "10:00", "12:00", 20) for d in ("Monday", "Wednesday", "Friday")]),
                ("vikas", "shah", "Cardiology", [(d, "14:00", "17:00", 30) for d in ("Tuesday", "Thursday")])]


Writing build/app/store.py


In [2]:
%%writefile -a build/app/store.py

@contextlib.contextmanager
def db(path):
    """One connection per operation. Commits on success, rolls back on error, always closes."""
    c = sqlite3.connect(path, timeout=5)
    c.row_factory = sqlite3.Row
    c.execute("PRAGMA foreign_keys = ON")
    try:
        with c:
            yield c
    finally:
        c.close()


def next_weekday(name: str, after: date | None = None) -> date:
    """The next `name` day strictly after `after` (default today)."""
    d = (after or date.today()) + timedelta(days=1)
    while d.strftime("%A") != name:
        d += timedelta(days=1)
    return d


def init(path, seed_demo: bool = False, reset: bool = False) -> Path:
    """Create the schema (idempotent) and, on an empty database and only if asked, demo data."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if reset:
        for suffix in ("", "-wal", "-shm"):
            Path(f"{path}{suffix}").unlink(missing_ok=True)
    with db(path) as c:
        c.execute("PRAGMA journal_mode = WAL")  # persists in the file
        c.executescript(SCHEMA)
        row = c.execute("SELECT value FROM Meta WHERE key='schema_version'").fetchone()
        if row and int(row[0]) > SCHEMA_VERSION:
            raise RuntimeError(f"database schema v{row[0]} is newer than this code (v{SCHEMA_VERSION})")
        c.execute("INSERT OR REPLACE INTO Meta VALUES ('schema_version', ?)", (str(SCHEMA_VERSION),))
        if seed_demo and not c.execute("SELECT 1 FROM Doctor LIMIT 1").fetchone():
            _seed(c)
    return path


def _seed(c) -> None:
    c.executemany("INSERT INTO Patient (First_Name, Last_Name, DOB, age, sex, Phone_Number) VALUES (?,?,?,?,?,?)",
                  DEMO_PATIENTS)
    for first, last, specialty, hours in DEMO_DOCTORS:
        doc = c.execute("INSERT INTO Doctor (First_Name, Last_Name, Specialty) VALUES (?,?,?)",
                        (first, last, specialty)).lastrowid
        c.executemany("INSERT INTO Availability VALUES (?,?,?,?,?)", [(doc, *h) for h in hours])
    tue, wed = next_weekday("Tuesday"), next_weekday("Wednesday")
    c.executemany("INSERT INTO Appointment (patient_id, doctor_id, appointment_date, status, reason, notes) "
                  "VALUES (?,?,?,?,?,?)", [
                      (1, 1, f"{tue} 10:00:00", "scheduled", "fever and cough", None),
                      (2, 2, f"{wed} 10:20:00", "scheduled", "vaccination", None),
                      (3, 1, f"{tue} 09:00:00", "cancelled", "back pain", None),
                      (None, 1, f"{wed} 00:00:00", "unavailable", None, "full-day leave")])


Appending to build/app/store.py


<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<h2 style="font-size:1.2em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">Patient rules</h2><p style="margin:.55em 0">Normalising phones, reading dates of birth, ages that must match.</p>

</div>

In [3]:
%%writefile build/app/domain/__init__.py
"""Business rules (DocAssist-derived), free of transport and model concerns."""


Writing build/app/domain/__init__.py


In [4]:
%%writefile build/app/domain/patients.py
"""Patient rules, after DocAssist (pyDocAssist: PatientService.register_patient, routes/patient_routes.py,
Utility.validate_*): names required, DOB as DD/MM/YYYY or YYYY-MM-DD and not in the future, age must match DOB
within 1 year, sex Male/Female, optional email must be valid, an adult's phone must be unique, phone stored with
the 91 prefix, names stored lower-case.
"""
import re
from datetime import date, datetime

from app.store import db

NAME_RE = re.compile(r"^[A-Za-z][A-Za-z .'-]{0,49}$")
EMAIL_RE = re.compile(r"^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$")  # DocAssist's regex


def normalize_phone(phone: str) -> str | None:
    """Indian mobile: 10 digits -> '91' + digits; 12 digits starting 91 kept. Anything else: None."""
    d = re.sub(r"\D", "", phone or "")
    if len(d) == 10:
        return "91" + d
    if len(d) == 12 and d.startswith("91"):
        return d
    return None


def parse_dob(s: str) -> date:
    for fmt in ("%d/%m/%Y", "%Y-%m-%d"):
        try:
            d = datetime.strptime(s.strip(), fmt).date()
            break
        except ValueError:
            continue
    else:
        raise ValueError("date of birth must be DD/MM/YYYY or YYYY-MM-DD")
    if d > date.today():
        raise ValueError("date of birth cannot be in the future")
    if d.year < 1900:
        raise ValueError("date of birth is before 1900")
    return d


def age_on(dob: date, today: date | None = None) -> int:
    t = today or date.today()
    return t.year - dob.year - ((t.month, t.day) < (dob.month, dob.day))


def dob_from_age(age: int, today: date | None = None) -> date:
    """DocAssist's estimate (same day, `age` years ago); 29 Feb falls back to 28 Feb."""
    t = today or date.today()
    try:
        return t.replace(year=t.year - age)
    except ValueError:
        return t.replace(year=t.year - age, day=28)


Writing build/app/domain/patients.py


<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<p style="margin:.55em 0"><code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">validate</code> is an allowlist over one registration; <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">duplicates</code> enforces "an adult's phone is unique".</p>

</div>

In [5]:
%%writefile -a build/app/domain/patients.py

def validate(first_name, last_name, phone, sex, dob="", age="", email="") -> tuple[dict | None, list[str]]:
    """Allowlist check of one registration. Returns (clean record, []) or (None, [problems])."""
    errs, rec = [], {}
    for key, v in (("first_name", first_name), ("last_name", last_name)):
        v = " ".join(str(v).split())
        if NAME_RE.fullmatch(v):
            rec[key] = v.lower()
        else:
            errs.append(f"{key.replace('_', ' ')}: letters only, 1-50 characters")
    rec["phone"] = normalize_phone(str(phone))
    if not rec["phone"]:
        errs.append("phone: must be a 10-digit mobile number (or 12 digits starting with 91)")
    if sex not in ("Male", "Female"):
        errs.append("sex: must be Male or Female")
    rec["sex"] = sex
    dob, age = str(dob).strip(), str(age).strip()
    born = years = None
    if dob:
        try:
            born = parse_dob(dob)
        except ValueError as e:
            errs.append(f"dob: {e}")
    if age:
        if age.isdigit() and int(age) <= 120:
            years = int(age)
        else:
            errs.append("age: a whole number from 0 to 120")
    if not dob and not age:
        errs.append("need a date of birth or an age")
    if born and years is not None and abs(age_on(born) - years) > 1:
        errs.append(f"age {years} doesn't match date of birth {born:%d %B %Y} (that is {age_on(born)})")
    if born:
        rec["dob"], rec["age"] = born, age_on(born)
    elif years is not None:
        rec["dob"], rec["age"] = dob_from_age(years), years
    email = str(email).strip()
    if email and not EMAIL_RE.fullmatch(email):
        errs.append("email: not a valid address")
    rec["email"] = email or None
    return (None, errs) if errs else (rec, [])


def duplicates(c, rec) -> list[str]:
    """DocAssist: an adult's phone must be unique; an email must be unique."""
    errs = []
    if rec["age"] >= 18:
        row = c.execute("SELECT First_Name, Last_Name FROM Patient WHERE Phone_Number=? AND is_active=1 AND age>=18",
                        (rec["phone"],)).fetchone()
        if row:
            errs.append(f"this phone is already registered to an adult patient ({row[0].title()} {row[1][0].upper()}.)")
    if rec["email"] and c.execute("SELECT 1 FROM Patient WHERE Email=? AND is_active=1", (rec["email"],)).fetchone():
        errs.append("this email is already registered to another patient")
    return errs


def check_new(path, rec) -> list[str]:
    with db(path) as c:
        return duplicates(c, rec)


Appending to build/app/domain/patients.py


<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<p style="margin:.55em 0">The read-back (names spelled, phone digit by digit) protects against mishearing; <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">insert</code> re-checks duplicates inside the write transaction.</p>

</div>

In [6]:
%%writefile -a build/app/domain/patients.py

def spell(word: str) -> str:
    return "-".join(ch.upper() for ch in word if ch.isalpha())


def read_back(rec) -> dict:
    """What the model should read out: spelled names, phone digit by digit."""
    return {"name": f"{rec['first_name'].title()} {rec['last_name'].title()}",
            "spelled": f"{spell(rec['first_name'])}  {spell(rec['last_name'])}",
            "phone_digits": " ".join(rec["phone"][2:]),
            "sex": rec["sex"],
            "date_of_birth": f"{rec['dob']:%d %B %Y}", "age": rec["age"],
            **({"email": rec["email"]} if rec["email"] else {})}


def insert(path, rec) -> dict:
    """Write a validated registration, re-checking duplicates inside the write transaction."""
    with db(path) as c:
        c.execute("BEGIN IMMEDIATE")
        if errs := duplicates(c, rec):
            return {"status": "invalid", "problems": errs}
        cur = c.execute("INSERT INTO Patient (First_Name, Last_Name, DOB, age, sex, Phone_Number, Email) "
                        "VALUES (?, ?, ?, ?, ?, ?, ?)",
                        (rec["first_name"], rec["last_name"], rec["dob"].isoformat(), rec["age"], rec["sex"],
                         rec["phone"], rec["email"]))
    return {"status": "registered", "patient_id": cur.lastrowid, "name": read_back(rec)["name"]}


def by_phone(path, phone: str) -> list[dict]:
    """Active patients on a phone (a parent and children may share one)."""
    with db(path) as c:
        rows = c.execute("SELECT Patient_ID, First_Name, Last_Name FROM Patient WHERE Phone_Number=? AND is_active=1 "
                         "ORDER BY Patient_ID", (phone,)).fetchall()
    return [{"id": r[0], "name": f"{r[1].title()} {r[2].title()}"} for r in rows]


Appending to build/app/domain/patients.py


<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<h2 style="font-size:1.2em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">Appointment rules</h2>
<p style="margin:.55em 0">Weekly hours in fixed slots; a slot is taken by any non-cancelled appointment or by leave; booking checks the date, the
slot, the reason and one booking per doctor per day; cancelling only touches the caller's own scheduled appointments.</p>

</div>

In [7]:
%%writefile build/app/domain/appointments.py
"""Appointment rules, after DocAssist (pyDocAssist: blueprints/appointment_api.py get_available_timeslots /
schedule_appointment / cancel_appointment, services/appointment_service.py):

- each doctor has weekly hours split into fixed-length slots (Availability.duration_minutes);
- a slot is taken by any non-cancelled appointment overlapping it;
- a doctor's "unavailable" entry closes the whole day ("full-day" in notes) or a range ("unavailable from HH:MM to HH:MM");
- booking: a doctor, a date that isn't past, a free slot start (not already over today), a reason;
  one active booking per patient per doctor per day (DocAssist flags this; a voice desk refuses it);
- cancelling: only the patient's own, only while "scheduled".
All functions take the database path and return plain dicts for the model.
"""
import re
from datetime import date, datetime

from app.store import db

STOP_WORDS = {"dr", "doctor", "the", "a", "with", "for", "and", "specialist"}


def fmt_time(minutes: int) -> str:
    h, m = divmod(minutes, 60)
    return f"{h % 12 or 12}:{m:02d} {'AM' if h < 12 else 'PM'}"


def fmt_day(d: date) -> str:
    return f"{d:%A} {d.day} {d:%B %Y}"


def parse_time(s: str) -> int | None:
    """'9:15 AM', '9 am', '09:15', '14:30', '2.30 pm' -> minutes after midnight; None if unreadable."""
    m = re.fullmatch(r"\s*(\d{1,2})(?:[:.](\d{2}))?\s*([ap])?\.?\s*m?\.?\s*", s.lower())
    if not m:
        return None
    h, mins, ap = int(m[1]), int(m[2] or 0), m[3]
    if mins > 59 or h > 23 or (ap and not 1 <= h <= 12):
        return None
    if ap:
        h = h % 12 + (12 if ap == "p" else 0)
    return h * 60 + mins


def parse_day(s: str) -> date | None:
    try:
        return datetime.strptime(s.strip(), "%Y-%m-%d").date()
    except ValueError:
        return None


Writing build/app/domain/appointments.py


<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<p style="margin:.55em 0">Doctors are found the way callers say them ("the pediatrician", "Dr Shah") but only ones that exist.</p>

</div>

In [8]:
%%writefile -a build/app/domain/appointments.py

def _doctors(c) -> list[dict]:
    out = []
    for r in c.execute("SELECT Doctor_ID, First_Name, Last_Name, Specialty FROM Doctor ORDER BY Doctor_ID"):
        days = [d[0] for d in c.execute("SELECT DISTINCT day FROM Availability WHERE doctor_id=?", (r[0],))]
        week = [d for d in ("Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday") if d in days]
        out.append({"id": r[0], "name": f"Dr. {r[1].title()} {r[2].title()}", "specialty": r[3], "days": week,
                    "_words": {r[1].lower(), r[2].lower(), *re.findall(r"[a-z]+", r[3].lower())}})
    return out


def doctors(path) -> list[dict]:
    with db(path) as c:
        return [{k: v for k, v in d.items() if not k.startswith("_")} for d in _doctors(c)]


def find_doctor(c, query: str) -> tuple[dict | None, dict | None]:
    """Resolve what the caller said ("Dr Kapoor", "the pediatrician", "cardiology") to exactly one doctor."""
    words = [w for w in re.findall(r"[a-z]+", query.lower()) if w not in STOP_WORDS]
    # prefixes, so "pediatrician" matches "pediatrics" and "cardio" matches "cardiology"
    hit = lambda d: any(dw.startswith(w[:6]) for w in words if len(w) >= 3 for dw in d["_words"])  # noqa: E731
    docs = _doctors(c)
    found = [d for d in docs if words and hit(d)]
    if len(found) == 1:
        return found[0], None
    names = [f"{d['name']} ({d['specialty']})" for d in docs]
    if not found:
        return None, {"status": "unknown_doctor", "problem": f"no doctor matches '{query}'", "doctors": names}
    return None, {"status": "which_doctor", "problem": f"'{query}' matches several doctors",
                  "doctors": [f"{d['name']} ({d['specialty']})" for d in found]}


Appending to build/app/domain/appointments.py


<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<p style="margin:.55em 0">Free slots, then booking inside one write transaction, so two callers can't take one slot.</p>

</div>

In [9]:
%%writefile -a build/app/domain/appointments.py

def _slots(c, doc: dict, day: date, now: datetime) -> dict:
    """Free and taken slot starts for one doctor and day, or why the day is closed."""
    if day < now.date():
        return {"closed": "that date is in the past"}
    hours = c.execute("SELECT start_time, end_time, duration_minutes FROM Availability WHERE doctor_id=? AND day=? "
                      "ORDER BY start_time", (doc["id"], f"{day:%A}")).fetchall()
    if not hours:
        return {"closed": f"{doc['name']} does not work on {day:%A}s (works {', '.join(doc['days'])})"}
    rows = c.execute("SELECT appointment_date, status, notes FROM Appointment WHERE doctor_id=? AND status!='cancelled' "
                     "AND substr(appointment_date, 1, 10)=?", (doc["id"], day.isoformat())).fetchall()
    busy, blocks = [], []
    for when, status, notes in rows:
        start = parse_time(when[11:16])
        notes = (notes or "").lower()
        if status == "unavailable" and "full-day" in notes:
            return {"closed": f"{doc['name']} is on leave that day"}
        if status == "unavailable" and (m := re.search(r"from\s(\d{2}:\d{2})\s*to\s*(\d{2}:\d{2})", notes)):
            blocks.append((parse_time(m[1]), parse_time(m[2])))
        elif status != "unavailable":
            busy.append(start)
    free, taken = [], []
    for start_s, end_s, dur in hours:
        m, end = parse_time(start_s), parse_time(end_s)
        while m + dur <= end:
            clash = any(m < b + dur and b < m + dur for b in busy) or any(a <= m < b for a, b in blocks)
            past = day == now.date() and m <= now.hour * 60 + now.minute
            (taken if clash or past else free).append(m)
            m += dur
    return {"free": free, "taken": taken}


def find_slots(path, doctor: str, day_s: str, now: datetime | None = None) -> dict:
    now = now or datetime.now()
    day = parse_day(day_s)
    if not day:
        return {"status": "bad_date", "problem": "date must be YYYY-MM-DD"}
    with db(path) as c:
        doc, problem = find_doctor(c, doctor)
        if problem:
            return problem
        s = _slots(c, doc, day, now)
    if "closed" in s:
        return {"status": "closed", "doctor": doc["name"], "date": fmt_day(day), "problem": s["closed"]}
    return {"status": "ok", "doctor": doc["name"], "specialty": doc["specialty"], "date": fmt_day(day),
            "free": [fmt_time(m) for m in s["free"]], "free_count": len(s["free"]),
            "next": "If the caller already asked for a time that is in free, confirm that one; otherwise offer two "
                    "or three free times. Never read the whole list."}


def book(path, patient_id: int, doctor: str, day_s: str, time_s: str, reason: str,
         now: datetime | None = None) -> dict:
    now = now or datetime.now()
    day, minutes, reason = parse_day(day_s), parse_time(time_s), " ".join(str(reason).split())[:200]
    if not day:
        return {"status": "bad_date", "problem": "date must be YYYY-MM-DD"}
    if minutes is None:
        return {"status": "bad_time", "problem": "time must look like 9:15 AM or 14:30"}
    if not reason:
        return {"status": "need_reason", "problem": "ask what the visit is for"}
    with db(path) as c:
        c.execute("BEGIN IMMEDIATE")  # check + insert in one write transaction: two callers can't take one slot
        doc, problem = find_doctor(c, doctor)
        if problem:
            return problem
        s = _slots(c, doc, day, now)
        if "closed" in s:
            return {"status": "closed", "problem": s["closed"]}
        if minutes not in s["free"]:
            return {"status": "slot_unavailable", "problem": f"{fmt_time(minutes)} is not free",
                    "free": [fmt_time(m) for m in s["free"]][:6]}
        same = c.execute("SELECT appointment_date FROM Appointment WHERE patient_id=? AND doctor_id=? AND "
                         "status='scheduled' AND substr(appointment_date, 1, 10)=?",
                         (patient_id, doc["id"], day.isoformat())).fetchone()
        if same:
            return {"status": "already_booked",
                    "problem": f"already booked with {doc['name']} that day at {fmt_time(parse_time(same[0][11:16]))}"}
        h, m = divmod(minutes, 60)
        appt = c.execute("INSERT INTO Appointment (patient_id, doctor_id, appointment_date, status, reason) "
                         "VALUES (?, ?, ?, 'scheduled', ?)",
                         (patient_id, doc["id"], f"{day} {h:02d}:{m:02d}:00", reason)).lastrowid
    return {"status": "booked", "appointment_id": appt, "doctor": doc["name"], "date": fmt_day(day),
            "time": fmt_time(minutes), "reason": reason}


Appending to build/app/domain/appointments.py


In [10]:
%%writefile -a build/app/domain/appointments.py

def upcoming(path, patient_id: int, now: datetime | None = None) -> list[dict]:
    now = now or datetime.now()
    with db(path) as c:
        rows = c.execute("SELECT a.Appointment_ID, a.appointment_date, a.reason, d.First_Name, d.Last_Name, d.Specialty "
                         "FROM Appointment a JOIN Doctor d ON d.Doctor_ID = a.doctor_id WHERE a.patient_id=? AND "
                         "a.status='scheduled' AND a.appointment_date>=? ORDER BY a.appointment_date",
                         (patient_id, f"{now:%Y-%m-%d %H:%M:00}")).fetchall()
    return [{"appointment_id": r[0], "doctor": f"Dr. {r[3].title()} {r[4].title()}", "specialty": r[5],
             "date": fmt_day(parse_day(r[1][:10])), "time": fmt_time(parse_time(r[1][11:16])), "reason": r[2]}
            for r in rows]


def cancel(path, patient_id: int, appointment_id: int, now: datetime | None = None) -> dict:
    now = now or datetime.now()
    with db(path) as c:
        r = c.execute("SELECT status, appointment_date FROM Appointment WHERE Appointment_ID=? AND patient_id=?",
                      (appointment_id, patient_id)).fetchone()
        if not r:  # someone else's appointment looks exactly like a missing one
            return {"status": "not_found", "problem": f"no appointment {appointment_id} for this patient"}
        if r[0] != "scheduled" or r[1] < f"{now:%Y-%m-%d %H:%M:00}":
            return {"status": "not_cancellable", "problem": f"appointment {appointment_id} is {r[0]} or already past"}
        c.execute("UPDATE Appointment SET status='cancelled' WHERE Appointment_ID=?", (appointment_id,))
    return {"status": "cancelled", "appointment_id": appointment_id}


if __name__ == "__main__":  # self-check: python -m app.domain.appointments
    import tempfile
    from pathlib import Path
    from app.store import init, next_weekday

    assert [parse_time(s) for s in ("9:15 AM", "9 am", "09:15", "14:30", "2.30 pm", "12:00 PM", "12 am")] == \
        [555, 540, 555, 870, 870, 720, 0]
    assert parse_time("25:00") is None and parse_time("13 pm") is None and parse_time("noon") is None
    assert fmt_time(555) == "9:15 AM" and fmt_time(870) == "2:30 PM" and fmt_time(0) == "12:00 AM"

    with tempfile.TemporaryDirectory() as d:
        db_ = init(Path(d) / "t.db", seed_demo=True)
        now = datetime.combine(date.today(), datetime.min.time()).replace(minute=1)  # the seed is relative to today
        tue, wed, thu = next_weekday("Tuesday"), next_weekday("Wednesday"), next_weekday("Thursday")
        with db(db_) as c:
            assert find_doctor(c, "Dr. Kapoor")[0]["name"] == "Dr. Anil Kapoor"
            assert find_doctor(c, "the pediatrician")[0]["name"] == "Dr. Sunita Rao"
            assert find_doctor(c, "cardiology")[0]["name"] == "Dr. Vikas Shah"
            assert find_doctor(c, "Dr. Smith")[1]["status"] == "unknown_doctor"
        s = find_slots(db_, "Kapoor", f"{tue}", now)  # 09:00-13:00 in 15 min = 16 slots, Asha holds 10:00
        assert s["status"] == "ok" and s["free_count"] == 15 and "10:00 AM" not in s["free"] and s["free"][0] == "9:00 AM"
        assert find_slots(db_, "Kapoor", f"{wed}", now)["status"] == "closed"            # full-day leave
        assert "does not work" in find_slots(db_, "Shah", f"{wed}", now)["problem"]       # cardiology: Tue/Thu only
        assert find_slots(db_, "Kapoor", "2000-01-03", now)["problem"] == "that date is in the past"
        late = datetime.combine(tue, datetime.min.time()).replace(hour=11, minute=5)       # same day: past slots taken
        assert find_slots(db_, "Kapoor", f"{tue}", late)["free"][0] == "11:15 AM"
        assert book(db_, 3, "Kapoor", f"{tue}", "10:00 AM", "checkup", now)["status"] == "slot_unavailable"
        assert book(db_, 3, "Kapoor", f"{tue}", "10:05 AM", "checkup", now)["status"] == "slot_unavailable"  # not a slot start
        assert book(db_, 3, "Kapoor", f"{tue}", "10:15", "", now)["status"] == "need_reason"
        ok = book(db_, 3, "Kapoor", f"{tue}", "10:15", "checkup", now)
        assert ok["status"] == "booked" and ok["time"] == "10:15 AM"
        assert book(db_, 3, "Kapoor", f"{tue}", "11:00 AM", "again", now)["status"] == "already_booked"
        assert book(db_, 1, "Kapoor", f"{tue}", "10:15 AM", "x", now)["status"] == "slot_unavailable"  # Vikram has it
        assert [a["time"] for a in upcoming(db_, 3, now)] == ["10:15 AM"]              # his cancelled seed isn't listed
        assert cancel(db_, 1, ok["appointment_id"], now)["status"] == "not_found"       # Asha can't cancel Vikram's
        assert cancel(db_, 3, ok["appointment_id"], now)["status"] == "cancelled"
        assert cancel(db_, 3, ok["appointment_id"], now)["status"] == "not_cancellable"
        assert book(db_, 1, "Kapoor", f"{tue}", "10:15 AM", "x", now)["status"] == "already_booked"  # Asha: 10:00 that day
        assert book(db_, 2, "Kapoor", f"{tue}", "10:15 AM", "x", now)["status"] == "booked"  # freed slot is bookable
        assert find_slots(db_, "Rao", f"{thu}", now)["status"] == "closed"
    print("ok")


Appending to build/app/domain/appointments.py


<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<h2 style="font-size:1.2em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">Numbers the caller said</h2>
<p style="margin:.55em 0">A speech-to-speech model hearing digits directly repeats or drops one (9845012367 heard as 98450012367, three
calls running), while a transcript of the same audio has them right. These rules read numbers from the caller's own
words, in Hindi, Tamil or English, and correct a phone number or code only when the model's value is impossible and
the words hold exactly one that fits.</p>

</div>

In [11]:
%%writefile build/app/domain/numbers.py
"""Numbers the caller said, from their own words (the transcript), in Hindi, Tamil or English.

Why: a speech-to-speech model hearing digits directly repeats or drops one (measured: 9845012367 heard as
98450012367, 8130653310 as 81306553310, three times running), while the transcript of the same audio had them
right. So the transcript is the evidence: `fix_digits` replaces a phone number or code the model got wrong with
the one the caller said. It only steps in when the model's value is impossible (wrong length) and the caller's
words hold exactly one number that fits, so it never overrides a plausible value with a guess.
"""
import re

# Spoken digits 0-9 as callers say them: English, Hindi, English words written in Devanagari, Tamil (formal and
# everyday). ponytail: single digits only ("बारह" = 12 stays a word)
DIGIT_WORDS = [
    "zero oh शून्य शुन्य जीरो ज़ीरो பூஜ்ஜியம் பூஜ்யம் சைபர் ஜீரோ",
    "one एक वन ஒன்று ஒன்னு",
    "two दो टू ரெண்டு இரண்டு",
    "three तीन थ्री மூன்று மூணு",
    "four चार फोर फ़ोर நான்கு நாலு",
    "five पाँच पांच फाइव फ़ाइव ஐந்து அஞ்சு",
    "six छह छः छै छे सिक्स ஆறு",
    "seven सात सेवन ஏழு",
    "eight आठ एट எட்டு",
    "nine नौ नाइन ஒன்பது ஒம்பது",
]
DIGIT = {w: str(d) for d, words in enumerate(DIGIT_WORDS) for w in words.split()}
REPEAT = {"double": 2, "डबल": 2, "டபுள்": 2, "triple": 3, "ट्रिपल": 3, "ட்ரிபிள்": 3}
_d = "|".join(sorted(DIGIT, key=len, reverse=True)) + r"|\d+"
_unit = rf"(?:(?:{'|'.join(REPEAT)})\s*)?(?:{_d})"
_edge = r"\wऀ-ॣ०-෿"  # letters + Indic letters and vowel signs (not the । ॥ full stops)
DIGIT_RUN = re.compile(rf"(?<![{_edge}]){_unit}(?:[\s,–-]+{_unit})*(?![{_edge}])", re.IGNORECASE)
DIGIT_UNIT = re.compile(rf"(?:({'|'.join(REPEAT)})\s*)?({_d})", re.IGNORECASE)

# Arguments we check, and the lengths a real value can have
VALID = {"phone": (10, 12), "code": (6,)}


def spoken_digits(text: str) -> str:
    """Numbers said digit by digit become digits: "नौ आठ चार पाँच" -> "9845", "nine double eight" -> "988",
    "81306 5 double 3 10" -> "8130653310". Runs shorter than 3 digits stay as said; runs already in plain digits
    ("10, 11") are left alone."""
    def join(m):
        units = DIGIT_UNIT.findall(m[0])
        if not any(w.lower() in DIGIT or r for r, w in units):
            return m[0]
        digits = "".join(DIGIT.get(w.lower(), w) * REPEAT.get(r.lower(), 1) for r, w in units)
        return digits if len(digits) >= 3 else m[0]
    return DIGIT_RUN.sub(join, text)


def caller_numbers(text: str) -> list[str]:
    """Every number in the caller's words, digit groups joined ("98450 12367" -> "9845012367")."""
    return [re.sub(r"\D", "", g) for g in re.findall(r"\d[\d\s–-]*\d|\d", spoken_digits(text))]


def fix_digits(args: dict, said: str) -> dict[str, tuple[str, str]]:
    """Correct `args` in place from the caller's words. Returns {argument: (model's value, caller's value)}."""
    fixed = {}
    heard = caller_numbers(said)
    for key, lengths in VALID.items():
        v = args.get(key)
        if not isinstance(v, str) or len(re.sub(r"\D", "", v)) in lengths:
            continue
        fits = {n for n in heard if len(n) in lengths}
        if len(fits) == 1:
            args[key] = fits.pop()
            fixed[key] = (v, args[key])
    return fixed


if __name__ == "__main__":
    assert spoken_digits("मेरा नंबर है नौ आठ चार पाँच, शून्य एक दो, तीन छः सात।") == "मेरा नंबर है 9845012367।"
    assert spoken_digits("एट वन थ्री जीरो सिक्स फाइव डबल थ्री वन जीरो।") == "8130653310।"
    assert caller_numbers("एट वन थ्री जीरो सिक्स five three three one zero") == ["8130653310"]
    assert caller_numbers("81306 5 double 3 10") == ["8130653310"]
    assert caller_numbers("slots at 10, 11, 12") == ["10", "11", "12"]  # commas separate numbers
    assert spoken_digits("मुझे एक अपॉइंटमेंट चाहिए") == "मुझे एक अपॉइंटमेंट चाहिए"
    a = {"phone": "81306553310"}
    assert fix_digits(a, "एट वन थ्री जीरो सिक्स फाइव डबल थ्री वन जीरो") == {"phone": ("81306553310", "8130653310")}
    a = {"phone": "9845012367"}                     # plausible: never overridden
    assert fix_digits(a, "nine eight four five zero one two three six six") == {} and a["phone"] == "9845012367"
    a = {"code": "49738"}
    assert fix_digits(a, "कोड है चार नौ सात तीन आठ आठ") == {"code": ("49738", "497388")}
    a = {"phone": "123"}                             # nothing that fits in the words: left for the tool to refuse
    assert fix_digits(a, "I don't remember") == {}
    print("numbers ok")


Writing build/app/domain/numbers.py


In [12]:
!cd build && python -m app.domain.numbers

numbers ok


<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<h2 style="font-size:1.2em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">Try it</h2>

</div>

In [13]:
import sys
sys.path.insert(0, 'build')   # `import app` now means the app you are building

In [14]:
from app import store
from app.domain import appointments, patients

db = store.init('build/data/step3.db', seed_demo=True, reset=True)
tue = store.next_weekday('Tuesday')
patients.validate('Ravi', 'Kumar', '98765 43210', 'Male', dob='15/08/1990')

({'first_name': 'ravi',
  'last_name': 'kumar',
  'phone': '919876543210',
  'sex': 'Male',
  'dob': datetime.date(1990, 8, 15),
  'age': 36,
  'email': None},
 [])

In [15]:
s = appointments.find_slots(db, 'Dr Kapoor', f'{tue}')
s['free_count'], s['free'][:4]

(15, ['9:00 AM', '9:15 AM', '9:30 AM', '9:45 AM'])

In [16]:
[appointments.book(db, 3, 'Kapoor', f'{tue}', '10:00 AM', 'checkup')['status'],   # Asha holds 10:00
 appointments.book(db, 3, 'Kapoor', f'{tue}', '10:15 AM', 'checkup')['status'],
 appointments.cancel(db, 1, 2)['status']]                                        # Rohan's, not Asha's

['slot_unavailable', 'booked', 'not_found']

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<p style="margin:.55em 0">The appointment module carries its own self-check:</p>

</div>

In [17]:
import subprocess
r = subprocess.run([sys.executable, '-m', 'app.domain.appointments'], cwd='build', capture_output=True, text=True)
r.stdout.strip() or r.stderr[-800:]

'ok'

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<p style="margin:.55em 0"><b>Next:</b> <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">04_skills.ipynb</code>: the functions the model calls.</p>

</div>